# Virtual Environments & Packages
**Topic:** Tools & Environment

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** why virtual environments are necessary by describing what happens without them
- **Describe** the difference between conda and pip and when each is appropriate
- **Interpret** an `environment.yml` or `requirements.txt` file as a reproducibility contract

> **Tip:** Start by reading the **environment isolation diagram** below before the explanation, and try to trace what happens when two projects need different versions of the same package.

---
## How we got here

In *02: Jupyter Notebooks* we ran Python code in cells and saw that the kernel holds all variables. But which Python is the kernel running? Which version of pandas does it see? These questions become urgent the moment you work on more than one project. Virtual environments are the answer.

---
## Why this matters for data science

Every data science job posting lists reproducibility as a requirement, and virtual environments are the tool that makes it possible. A model trained with `scikit-learn==1.3.0` may produce subtly different results with `scikit-learn==1.5.0`. A pipeline that works on your laptop with Python 3.11 may fail on the server running Python 3.9. Pinning your environment in a file that teammates and servers can recreate exactly is the professional standard.

---
## Try it yourself

In [ ]:
# Widget: Two side-by-side project cards (Project A and Project B)
# Each card shows its required packages and versions
#   Project A: pandas==1.5.3, scikit-learn==1.2.2, python=3.10
#   Project B: pandas==2.1.4, scikit-learn==1.5.1, python=3.12
# A toggle switches between "No environments (shared system Python)" and "With conda environments"
# In "no environments" mode: a warning banner appears when the user clicks "Install Project B"
#   showing that installing pandas 2.1.4 overwrites the 1.5.3 needed by Project A
# In "with environments" mode: both projects coexist with no conflict
# Student should notice: the same package at different versions cannot coexist
#   in one Python installation; environments solve this by giving each project its own Python

---
## What's happening?

A virtual environment is an isolated directory containing its own Python interpreter and its own set of installed packages. Activating one tells your terminal to use that directory's Python instead of the system one.

```
my-ds-project/
├── .conda/                  ← the environment lives here (or outside the project)
│   ├── bin/
│   │   └── python           ← this project's Python executable
│   └── lib/
│       └── python3.12/
│           └── site-packages/
│               ├── pandas/  ← pandas installed only for this environment
│               └── sklearn/ ← scikit-learn installed only for this environment
├── notebooks/
│   └── analysis.ipynb
└── environment.yml          ← the recipe to recreate this environment
```

| Tool | Creates environments using | Package manager | Best for |
|------|--------------------------|----------------|----------|
| `conda` | `conda create -n name python=3.12` | `conda` + `pip` | Data science (handles non-Python C dependencies like BLAS, CUDA) |
| `venv` | `python -m venv env/` | `pip` only | Pure Python projects without heavy binary dependencies |
| `poetry` | `poetry init` | `poetry` | Modern projects with strict dependency resolution |

```bash
# The standard data science environment workflow
conda create -n ds_env python=3.12
conda activate ds_env
pip install pandas scikit-learn plotly ipywidgets jupyter

# Export for reproducibility
conda env export > environment.yml

# Recreate on another machine or share with a teammate
conda env create -f environment.yml
conda activate ds_env
```

### Two environments, zero conflict

```
┌─────────────────────────────┐    ┌─────────────────────────────┐
│  ENV: project_a             │    │  ENV: project_b             │
│  Python 3.10                │    │  Python 3.12                │
│  pandas    == 1.5.3  ✓      │    │  pandas    == 2.1.4  ✓      │
│  sklearn   == 1.2.2  ✓      │    │  sklearn   == 1.5.1  ✓      │
│  plotly    == 5.14.1 ✓      │    │  plotly    == 5.24.0 ✓      │
└─────────────────────────────┘    └─────────────────────────────┘
         ↑                                    ↑
  conda activate project_a          conda activate project_b
  (completely isolated)             (completely isolated)
```

Return to the widget and click through the "no environments" scenario to see the conflict appear, then switch to "with environments" to see it disappear.

---
## Real-world example: Two data science projects on the same laptop

The chart shows a realistic scenario: a data science team maintains two active projects with conflicting scikit-learn version requirements. Without environments, installing one project's dependencies breaks the other.

Notice:

- **Notice:** The conflict is not hypothetical; scikit-learn changed the default value of `max_iter` in `LogisticRegression` between 1.2 and 1.5, meaning a model trained on one version will produce a convergence warning (or different coefficients) on the other
- **Notice:** The fix is not to pin one version globally but to give each project its own isolated environment where both versions can coexist on the same machine simultaneously
- **Notice:** The `environment.yml` file in each project folder is the contract; anyone who clones the project and runs `conda env create -f environment.yml` gets an identical environment, making results reproducible

> **Discussion question:** A teammate argues that using Docker containers achieves the same isolation as conda environments and is more portable. They are right that Docker is more portable. What does conda give you that Docker does not, and when would you use each?

In [ ]:
# ── Conflict scenario: two projects, one environment vs two environments ──────
scenarios = [
    "Without environments:
Project A installs sklearn 1.2",
    "Without environments:
Project B installs sklearn 1.5 (breaks A)",
    "With environments:
Project A env — sklearn 1.2",
    "With environments:
Project B env — sklearn 1.5",
]
test_accuracy = [0.847, 0.831, 0.847, 0.843]  # A degrades without environments
colors = [
    PALETTE["muted"],
    PALETTE["secondary"],
    PALETTE["accent"],
    PALETTE["primary"],
]

fig = go.Figure(go.Bar(
    x=scenarios,
    y=test_accuracy,
    marker_color=colors,
    text=[f"{v:.1%}" for v in test_accuracy],
    textposition="outside",
))
layout = base_layout(
    title="Model Accuracy Under Different Environment Strategies",
    xaxis_title="",
    yaxis_title="Test Accuracy (Logistic Regression on same dataset)",
)
layout.update(yaxis=dict(range=[0.80, 0.87], tickformat=".0%"))
fig.update_layout(layout)
fig.show()

### Common environment scenarios and outcomes

| Scenario | Without environments | With environments |
|----------|--------------------|--------------------|
| Two projects need different pandas versions | Installing one overwrites the other; at least one project breaks | Each project gets its own pandas version; both work simultaneously |
| New team member joins the project | "Works on my machine": no guarantee theirs matches | `conda env create -f environment.yml` recreates the exact environment |
| Moving from laptop to cloud server | Code may fail due to different system Python or library versions | Same `environment.yml` produces identical environment on any machine |
| Upgrading a package to test compatibility | Upgrade affects all projects on the system | Upgrade inside one environment; other environments are untouched |
| Returning to a project after 6 months | Libraries may have updated; old code may break | Frozen `environment.yml` restores exact versions from 6 months ago |

---
## Key takeaway

> **A virtual environment is an isolated box containing a specific Python and pinned packages; exporting it to `environment.yml` is the one-line contract that makes your results reproducible on any machine.**

---
*Next up: Command Line Basics — the text interface you will use to create environments, run scripts, and navigate your file system in every data science workflow*